#  Sistem Pakar Diagnosa Penyakit Gastro Usus

**Mata Kuliah:** Sistem Pakar  
**Tugas:** Praktikum 1 – Sistem Pakar Identifikasi Infeksi Gastro-Usus  
**Tools:** Python (Pure Rule-Based Expert System)

---

## Tujuan
Membangun sistem pakar berbasis aturan (rule-based expert system) untuk mendiagnosa kemungkinan penyakit infeksi sistem Gastro-Usus berdasarkan gejala yang dialami pengguna, dengan menghitung persentase kemungkinan (confidence) setiap diagnosis.

---

## Latar Belakang

Sistem pakar (Expert System) adalah program komputer yang meniru kemampuan seorang pakar dalam bidang tertentu. Sistem ini terdiri dari:
- **Knowledge Base**: Basis pengetahuan berisi fakta dan aturan (rules)
- **Inference Engine**: Mesin inferensi yang memproses aturan berdasarkan fakta
- **User Interface**: Antarmuka untuk interaksi dengan pengguna

Pada praktikum ini digunakan **Forward Chaining**: dimulai dari fakta (gejala) → menarik kesimpulan (diagnosis).

---

## Kategori Penyakit

| No | Kode | Nama Penyakit |
|----|------|---------------|
| 1  | P33  | Keracunan Staphylococcus aureus |
| 2  | P34  | Keracunan Jamur Beracun |
| 3  | P35  | Keracunan Salmonellae |
| 4  | P36  | Keracunan Clostridium botulinum |
| 5  | P37  | Keracunan Campylobacter |

---
##  Bagian 1: Knowledge Base (Basis Pengetahuan)

Knowledge base berisi:
1. **Daftar gejala** (symptoms) — kode dan deskripsi pertanyaan kepada user
2. **Aturan diagnosis** (rules) — penyakit beserta gejala-gejala khasnya

Aturan dimodelkan dalam bentuk:
```
IF gejala_1 AND gejala_2 AND ... THEN diagnosis_X dengan confidence Y%
```

In [ ]:
# ============================================================
# KNOWLEDGE BASE — Basis Pengetahuan Sistem Pakar Gastro Usus
# ============================================================

# Daftar gejala (symptom_id: deskripsi pertanyaan)
# Nomor mengacu pada indeks gejala dari soal praktikum
GEJALA = {
    'G01': 'Apakah Anda sering buang air besar (lebih dari 2 kali)?',
    'G02': 'Apakah Anda mengalami berak encer (mencret)?',
    'G03': 'Apakah Anda mengalami berak berdarah?',
    'G04': 'Apakah Anda merasa lesu dan tidak bergairah?',
    'G05': 'Apakah Anda tidak selera makan?',
    'G06': 'Apakah Anda merasa mual dan sering muntah (lebih dari 1 kali)?',
    'G07': 'Apakah Anda merasa sakit di bagian perut?',
    'G08': 'Apakah tekanan darah Anda rendah?',
    'G09': 'Apakah Anda merasa pusing?',
    'G10': 'Apakah Anda pernah pingsan?',
    'G11': 'Apakah suhu badan Anda tinggi (demam)?',
    'G12': 'Apakah Anda memiliki luka di bagian tertentu?',
    'G13': 'Apakah Anda tidak dapat menggerakkan anggota badan tertentu (lumpuh)?',
    'G14': 'Apakah Anda baru saja memakan sesuatu yang mencurigakan?',
    'G15': 'Apakah Anda memakan daging (terutama daging yang tidak dimasak sempurna)?',
    'G16': 'Apakah Anda memakan jamur?',
    'G17': 'Apakah Anda memakan makanan kaleng?',
    'G18': 'Apakah Anda membeli atau meminum susu?',
}

# ============================================================
# RULES — Aturan Diagnosis (Knowledge Representation)
# Setiap penyakit didefinisikan dengan:
#   - nama: nama penyakit
#   - gejala_utama: gejala yang WAJIB ada (bobot tinggi)
#   - gejala_pendukung: gejala tambahan yang memperkuat diagnosis
#   - total_gejala: total gejala yang dipertimbangkan (untuk menghitung persentase)
# ============================================================

RULES = {
    'P33': {
        'nama': 'Keracunan Staphylococcus aureus',
        'deskripsi': 'Keracunan yang disebabkan oleh toksin bakteri Staphylococcus aureus '
                    'pada makanan. Biasanya terjadi 1-6 jam setelah makan makanan yang terkontaminasi.',
        'gejala_utama':     ['G06', 'G07', 'G14'],   # muntah, sakit perut, makan sesuatu
        'gejala_pendukung': ['G01', 'G02', 'G04', 'G05', 'G08', 'G09'],
        'saran': 'Istirahat, minum banyak cairan untuk cegah dehidrasi. Jika muntah berlebihan, segera ke dokter.'
    },
    'P34': {
        'nama': 'Keracunan Jamur Beracun',
        'deskripsi': 'Keracunan akibat memakan jamur beracun. Gejalanya dapat muncul '
                    'beberapa jam hingga beberapa hari setelah konsumsi.',
        'gejala_utama':     ['G16', 'G06', 'G07'],   # makan jamur, muntah, sakit perut
        'gejala_pendukung': ['G01', 'G02', 'G04', 'G05', 'G08', 'G09', 'G10', 'G11'],
        'saran': 'SEGERA ke UGD rumah sakit! Keracunan jamur bisa berbahaya. Bawa sisa jamur yang dimakan jika ada.'
    },
    'P35': {
        'nama': 'Keracunan Salmonellae',
        'deskripsi': 'Infeksi bakteri Salmonella yang sering berasal dari daging, '
                    'telur, atau produk susu yang tidak matang/tidak higienis.',
        'gejala_utama':     ['G15', 'G01', 'G11'],   # makan daging, mencret, demam
        'gejala_pendukung': ['G02', 'G03', 'G06', 'G07', 'G04', 'G08', 'G12'],
        'saran': 'Konsumsi oralit untuk mencegah dehidrasi, istirahat cukup. Segera ke dokter jika ada darah pada tinja atau demam tinggi.'
    },
    'P36': {
        'nama': 'Keracunan Clostridium botulinum',
        'deskripsi': 'Keracunan serius akibat toksin bakteri Clostridium botulinum '
                    'yang umumnya berasal dari makanan kaleng yang tidak diproses dengan benar.',
        'gejala_utama':     ['G17', 'G13', 'G06'],   # makan kaleng, lumpuh, muntah
        'gejala_pendukung': ['G01', 'G07', 'G08', 'G09', 'G10', 'G04', 'G05'],
        'saran': 'DARURAT MEDIS! Segera ke IGD. Clostridium botulinum dapat menyebabkan kelumpuhan pernapasan yang mengancam jiwa.'
    },
    'P37': {
        'nama': 'Keracunan Campylobacter',
        'deskripsi': 'Infeksi bakteri Campylobacter, umumnya dari produk susu mentah '
                    'atau unggas yang tidak dimasak sempurna.',
        'gejala_utama':     ['G18', 'G03', 'G11'],   # minum susu, berak berdarah, demam
        'gejala_pendukung': ['G01', 'G02', 'G07', 'G04', 'G06', 'G09'],
        'saran': 'Istirahat, jaga hidrasi. Konsultasi dokter untuk pemberian antibiotik jika diperlukan.'
    },
}

print("✅ Knowledge base berhasil dimuat.")
print(f"   - Jumlah gejala  : {len(GEJALA)} gejala")
print(f"   - Jumlah penyakit: {len(RULES)} penyakit")

---
## ⚙️ Bagian 2: Inference Engine (Mesin Inferensi)

Inference engine menggunakan **Forward Chaining**:
1. Kumpulkan fakta dari user (gejala yang dialami)
2. Cocokkan fakta dengan setiap rule di knowledge base
3. Hitung confidence score untuk setiap penyakit:

$$\text{Confidence}(\%) = \frac{\text{Jumlah gejala cocok}}{\text{Total gejala rule}} \times 100$$

Gejala utama (mandatory) mendapat **bobot 2x** dibanding gejala pendukung.

4. Tentukan diagnosis berdasarkan threshold confidence

In [ ]:
# ============================================================
# INFERENCE ENGINE — Mesin Inferensi Forward Chaining
# ============================================================

def hitung_confidence(fakta_user: set, rule: dict) -> float:
    """
    Menghitung confidence score suatu penyakit berdasarkan fakta user.

    Metode:
    - Gejala utama yang cocok  -> bobot 2
    - Gejala pendukung cocok   -> bobot 1
    - Skor maksimum = (len(utama) * 2) + len(pendukung)
    """
    gejala_utama = set(rule['gejala_utama'])
    gejala_pendukung = set(rule['gejala_pendukung'])

    skor_max = len(gejala_utama) * 2 + len(gejala_pendukung)
    skor_utama = len(fakta_user & gejala_utama) * 2
    skor_pendukung = len(fakta_user & gejala_pendukung)
    skor_aktual = skor_utama + skor_pendukung

    if skor_max == 0:
        return 0.0

    return round((skor_aktual / skor_max) * 100, 2)


def inferensi(fakta_user: set, threshold: float = 30.0) -> tuple[list[dict], list[dict]]:
    """
    Melakukan inferensi forward chaining terhadap seluruh rules.

    Returns:
        tuple[list[dict], list[dict]]:
        - hasil_filtered: hasil di atas threshold
        - hasil: semua hasil untuk rekap/debug
    """
    hasil = []

    for kode, rule in RULES.items():
        confidence = hitung_confidence(fakta_user, rule)
        hasil.append({
            'kode': kode,
            'nama': rule['nama'],
            'deskripsi': rule['deskripsi'],
            'confidence': confidence,
            'saran': rule['saran'],
        })

    hasil.sort(key=lambda x: x['confidence'], reverse=True)
    hasil_filtered = [h for h in hasil if h['confidence'] >= threshold]
    return hasil_filtered, hasil


print("Inference engine berhasil dimuat.")

---
## Bagian 3: User Interface (CLI)

Notebook ini tetap menyimpan mode CLI agar logika konsultasi per pertanyaan bisa diuji.

Untuk penggunaan harian, versi terbaru sistem menggunakan **GUI berbasis browser** (dibahas di Bagian 6).

In [ ]:
# ============================================================
# USER INTERFACE — Antarmuka Interaktif
# ============================================================

def tampilkan_header():
    """Menampilkan header/judul sistem pakar."""
    print("=" * 65)
    print("  SISTEM PAKAR IDENTIFIKASI INFEKSI SISTEM GASTRO-USUS")
    print("  Berbasis Rule-Based Expert System | Forward Chaining")
    print("=" * 65)
    print()
    print("Sistem ini akan menanyakan beberapa gejala yang Anda alami.")
    print("Jawab dengan: y (ya) atau n (tidak)")
    print()


def kumpulkan_gejala() -> set:
    """
    Mengumpulkan fakta dari user melalui sesi tanya-jawab interaktif.
    
    Returns:
        set: Himpunan kode gejala yang dialami user
    """
    fakta = set()
    total = len(GEJALA)
    
    print(f"📋 Silakan jawab {total} pertanyaan berikut:")
    print("-" * 65)
    
    for i, (kode, pertanyaan) in enumerate(GEJALA.items(), start=1):
        while True:
            jawaban = input(f"[{i:02d}/{total}] {pertanyaan} (y/n): ").strip().lower()
            if jawaban in ('y', 'n', 'ya', 'tidak', 'yes', 'no'):
                break
            print("       ⚠️  Masukkan hanya 'y' atau 'n'")
        
        if jawaban in ('y', 'ya', 'yes'):
            fakta.add(kode)
    
    return fakta


def tampilkan_hasil(hasil_filtered: list, semua_hasil: list, threshold: float):
    """
    Menampilkan hasil diagnosis kepada pengguna.
    
    Args:
        hasil_filtered : Hasil diagnosis yang melewati threshold
        semua_hasil    : Semua hasil (untuk menampilkan rekap)
        threshold      : Nilai threshold yang digunakan
    """
    print()
    print("=" * 65)
    print("  📊 HASIL DIAGNOSIS")
    print("=" * 65)
    
    # Tampilkan semua persentase (ringkasan)
    print("\n📈 Rekap Confidence Semua Penyakit:")
    print("-" * 45)
    for h in semua_hasil:
        bar_len  = int(h['confidence'] / 5)  # max 20 karakter
        bar      = '█' * bar_len + '░' * (20 - bar_len)
        marker   = ' ✅' if h['confidence'] >= threshold else ''
        print(f"  {h['nama']:<40} : {h['confidence']:5.1f}%{marker}")
        print(f"  {bar}")
    
    print()
    print(f"  (Threshold: {threshold}%)")
    print("-" * 65)
    
    if not hasil_filtered:
        print()
        print("  ℹ️  Tidak ada diagnosis yang memenuhi threshold.")
        print("     Kemungkinan gejala yang Anda alami tidak spesifik.")
        print("     Disarankan untuk berkonsultasi langsung dengan dokter.")
    else:
        # Diagnosis utama (confidence tertinggi)
        utama = hasil_filtered[0]
        print()
        print(f"  🏆 Diagnosis Utama:")
        print(f"     ➤ {utama['nama']}")
        print(f"     Confidence: {utama['confidence']}%")
        print()
        print(f"  📖 Deskripsi:")
        # Wrap teks agar rapi
        import textwrap
        desc_wrapped = textwrap.fill(utama['deskripsi'], width=58,
                                     initial_indent='     ',
                                     subsequent_indent='     ')
        print(desc_wrapped)
        print()
        print(f"  💡 Saran:")
        saran_wrapped = textwrap.fill(utama['saran'], width=58,
                                      initial_indent='     ',
                                      subsequent_indent='     ')
        print(saran_wrapped)
        
        # Tampilkan kemungkinan lain jika ada
        if len(hasil_filtered) > 1:
            print()
            print("  ⚠️  Kemungkinan penyakit lain yang perlu dipertimbangkan:")
            for h in hasil_filtered[1:]:
                print(f"     • {h['nama']} ({h['confidence']}%)")
    
    print()
    print("=" * 65)
    print("  ⚕️  PERHATIAN: Hasil ini hanya bersifat informatif.")
    print("      Konsultasikan dengan dokter untuk diagnosis resmi.")
    print("=" * 65)


print("✅ User interface berhasil dimuat.")

---
## Bagian 4: Menjalankan Sistem Pakar

Tersedia dua mode:
1. **CLI**: fungsi `jalankan_sistem_pakar(threshold=30.0)`
2. **GUI Browser**: fungsi `jalankan_visualisasi_browser()`

Di file Python `.py`, mode default yang dijalankan adalah GUI browser.

In [ ]:
# ============================================================
# MAIN PROGRAM — Jalankan Sistem Pakar (CLI)
# ============================================================

def jalankan_sistem_pakar(threshold: float = 30.0):
    """
    Fungsi utama untuk menjalankan sistem pakar secara interaktif.

    Args:
        threshold (float): Batas minimum confidence untuk ditampilkan (%)
    """
    tampilkan_header()

    # Fase 1: Kumpulkan gejala dari user
    fakta_user = kumpulkan_gejala()

    print()
    print(f"  Gejala yang Anda tandai: {len(fakta_user)} dari {len(GEJALA)}")
    if fakta_user:
        gejala_terpilih = [GEJALA[k][:50] + '...' if len(GEJALA[k]) > 50 else GEJALA[k]
                           for k in sorted(fakta_user)]
        for g in gejala_terpilih:
            print(f"     - {g}")

    # Fase 2: Inferensi (forward chaining)
    print()
    print("  Memproses inference engine...")
    hasil_filtered, semua_hasil = inferensi(fakta_user, threshold)

    # Fase 3: Tampilkan hasil
    tampilkan_hasil(hasil_filtered, semua_hasil, threshold)


# Contoh pemanggilan manual di notebook:
# jalankan_sistem_pakar(threshold=30.0)

---
## Bagian 5: Pengujian Sistem (Opsional)

Bagian ini dipakai untuk memvalidasi rule dan confidence tanpa input manual.

Definisi fungsi disesuaikan dengan file `.py`, yaitu pengujian dijalankan melalui `jalankan_pengujian_skenario()` saat dibutuhkan.

In [ ]:
# ============================================================
# UNIT TESTING — Pengujian Skenario Gejala (opsional)
# ============================================================

def jalankan_pengujian_skenario() -> None:
    skenario_uji = [
        {
            'nama': 'Skenario A - Dugaan Staphylococcus',
            'gejala': {'G06', 'G07', 'G14', 'G04', 'G05'},
            'ekspektasi': 'P33'
        },
        {
            'nama': 'Skenario B - Dugaan Clostridium botulinum',
            'gejala': {'G17', 'G13', 'G06', 'G09', 'G08'},
            'ekspektasi': 'P36'
        },
        {
            'nama': 'Skenario C - Dugaan Campylobacter',
            'gejala': {'G18', 'G03', 'G11', 'G01', 'G07'},
            'ekspektasi': 'P37'
        },
        {
            'nama': 'Skenario D - Dugaan Salmonellae',
            'gejala': {'G15', 'G01', 'G11', 'G02', 'G06'},
            'ekspektasi': 'P35'
        },
        {
            'nama': 'Skenario E - Gejala Minim (tidak spesifik)',
            'gejala': {'G04'},
            'ekspektasi': None
        },
    ]

    print("=" * 65)
    print("  HASIL PENGUJIAN SKENARIO")
    print("=" * 65)

    for skenario in skenario_uji:
        hasil_filtered, _ = inferensi(skenario['gejala'], threshold=30.0)
        print(f"\n- {skenario['nama']}")
        if hasil_filtered:
            top = hasil_filtered[0]
            status = "BENAR" if top['kode'] == skenario['ekspektasi'] else "BERBEDA"
            print(f"  Top: {top['kode']} - {top['nama']} ({top['confidence']}%) [{status}]")
        else:
            status = "BENAR" if skenario['ekspektasi'] is None else "BERBEDA"
            print(f"  Top: Tidak ada diagnosis [{status}]")


# Contoh pemanggilan manual di notebook:
# jalankan_pengujian_skenario()

---
## Bagian 6: Visualisasi dan GUI Browser

Versi terbaru tidak lagi mengandalkan GUI desktop Tkinter.

Visualisasi antarmuka dibuat sebagai **HTML interaktif** yang otomatis dibuka di browser:
- Panel kiri kuning untuk checklist gejala
- Panel kanan abu-abu untuk hasil confidence
- Kontrol threshold dan diagnosis utama di panel bawah

In [ ]:
# ============================================================
# VISUALISASI — GUI Browser (HTML interaktif)
# ============================================================

def jalankan_visualisasi_browser() -> None:
    gejala_js = json.dumps(GEJALA, ensure_ascii=False)
    rules_js = json.dumps(RULES, ensure_ascii=False)

    html_content = f"""<!doctype html>
<html lang=\"id\">
<head>
  <meta charset=\"utf-8\">
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">
  <title>Sistem Pakar Gastro Usus</title>
  <style>
    body {{
      margin: 0;
      background: #d9d9d9;
      font-family: Tahoma, Arial, sans-serif;
      color: #111;
    }}
    .window {{
      width: min(1180px, calc(100vw - 24px));
      height: min(760px, calc(100vh - 24px));
      margin: 12px auto;
      border: 1px solid #777;
      background: #f2f2f2;
      display: grid;
      grid-template-columns: 64% 36%;
      grid-template-rows: 1fr auto;
      gap: 0;
    }}
    .left {{
      background: #fff44d;
      border-right: 1px solid #8a8a8a;
      overflow: hidden;
      display: flex;
      flex-direction: column;
    }}
    .left .list {{
      overflow-y: auto;
      padding: 8px 10px 10px;
    }}
    .right {{
      background: #efefef;
      overflow: hidden;
      display: flex;
      flex-direction: column;
    }}
    .right .result {{
      white-space: pre-wrap;
      font-size: 26px;
      line-height: 1.36;
      padding: 10px 12px;
      overflow-y: auto;
      margin: 0;
      font-family: Tahoma, Arial, sans-serif;
    }}
    .bottom {{
      grid-column: 1 / span 2;
      border-top: 1px solid #8a8a8a;
      background: #ddd;
      display: flex;
      align-items: center;
      gap: 10px;
      padding: 6px 10px;
      font-size: 24px;
      flex-wrap: wrap;
    }}
    .spacer {{
      flex: 1;
    }}
    .diagnosis {{
      color: #b10000;
      font-weight: bold;
    }}
    .row {{
      display: flex;
      align-items: center;
      gap: 6px;
      margin: 3px 0;
      font-size: 23px;
    }}
    input[type=\"checkbox\"] {{
      width: 18px;
      height: 18px;
      accent-color: #d9d9d9;
    }}
    input[type=\"number\"] {{
      width: 70px;
      height: 34px;
      font-size: 22px;
      padding: 2px 4px;
    }}
    button {{
      font-size: 22px;
      padding: 4px 14px;
      cursor: pointer;
    }}
  </style>
</head>
<body>
  <div class=\"window\">
    <section class=\"left\"><div id=\"list\" class=\"list\"></div></section>
    <section class=\"right\"><pre id=\"result\" class=\"result\">Pilih gejala lalu klik Proses.</pre></section>
    <section class=\"bottom\">
      <span>Threshold</span>
      <input id=\"threshold\" type=\"number\" min=\"0\" max=\"100\" value=\"30\">
      <span>%</span>
      <button id=\"proses\" type=\"button\">Proses</button>
      <span class=\"spacer\"></span>
      <span>Anda terkena infeksi:</span>
      <span id=\"top\" class=\"diagnosis\">-</span>
    </section>
  </div>

  <script>
    const GEJALA = {gejala_js};
    const RULES = {rules_js};

    const listEl = document.getElementById('list');
    const resultEl = document.getElementById('result');
    const topEl = document.getElementById('top');

    function renderGejala() {{
      Object.entries(GEJALA).forEach(([kode, pertanyaan]) => {{
        const row = document.createElement('label');
        row.className = 'row';
        row.innerHTML = `<input type=\"checkbox\" data-kode=\"${{kode}}\"> <span>${{pertanyaan}}</span>`;
        listEl.appendChild(row);
      }});
    }}

    function hitungConfidence(fakta, rule) {{
      const utama = new Set(rule.gejala_utama);
      const pendukung = new Set(rule.gejala_pendukung);
      const skorMax = (utama.size * 2) + pendukung.size;
      let skor = 0;
      fakta.forEach((kode) => {{
        if (utama.has(kode)) skor += 2;
        else if (pendukung.has(kode)) skor += 1;
      }});
      if (skorMax === 0) return 0;
      return Number(((skor / skorMax) * 100).toFixed(2));
    }}

    function proses() {{
      const threshold = Number(document.getElementById('threshold').value);
      if (Number.isNaN(threshold) || threshold < 0 || threshold > 100) {{
        alert('Threshold harus angka 0-100.');
        return;
      }}

      const fakta = new Set(
        Array.from(document.querySelectorAll('input[data-kode]:checked')).map((el) => el.dataset.kode)
      );

      const hasil = Object.entries(RULES).map(([kode, rule]) => {{
        const confidence = hitungConfidence(fakta, rule);
        return {{ kode, nama: rule.nama, confidence }};
      }}).sort((a, b) => b.confidence - a.confidence);

      const lines = hasil.map((h) => `${{h.nama.replace('Keracunan ', '')}} : ${{h.confidence.toFixed(2)}} %`);
      resultEl.textContent = lines.join('\\n');

      const top = hasil.find((h) => h.confidence >= threshold);
      topEl.textContent = top ? top.nama : 'Tidak terdeteksi';
    }}

    document.getElementById('proses').addEventListener('click', proses);
    renderGejala();
  </script>
</body>
</html>
"""

    output_html = Path.cwd() / 'sistem_pakar_gui_browser.html'
    output_html.write_text(html_content, encoding='utf-8')
    webbrowser.open(output_html.resolve().as_uri())
    print(f"GUI browser dibuka: {output_html.name}")


# Jalankan GUI browser:
# jalankan_visualisasi_browser()

---
## Bagian 7: Kesimpulan

### Alur Logika Sistem (Versi Terbaru)

```text
[Checklist Gejala di Browser]
          |
          v
[Fakta (Working Memory)]
          |
          v
[Inference Engine — Forward Chaining]
          |
          +-- Hitung confidence tiap penyakit (bobot utama 2x)
          |
          v
[Filter berdasarkan threshold]
          |
          v
[Output: daftar confidence + diagnosis utama]
```

### Rumus Confidence

$$
\text{Confidence}(\%) = \frac{(\text{Gejala Utama Cocok} \times 2) + (\text{Gejala Pendukung Cocok} \times 1)}{(\text{Total Utama} \times 2) + (\text{Total Pendukung} \times 1)} \times 100
$$

### Catatan Perubahan dari Versi Lama
- Visualisasi heatmap PNG/HTML tidak dipakai lagi sebagai mode utama.
- Antarmuka utama kini berupa GUI browser interaktif.
- Kode notebook diselaraskan dengan fungsi pada file `sistem_pakar_gastro_usus.py`.

### Keterbatasan
- Basis pengetahuan masih terbatas pada rule yang tersedia.
- Tidak mempertimbangkan komorbid/lebih dari satu penyakit sekaligus.
- Belum menggunakan certainty factor berbasis input keyakinan user.